In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/processed/student_placement_cleaned.csv")

print("Dataset Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

Dataset Shape: (12000, 13)

Columns:
['gender', 'age', 'degree', 'branch', 'cgpa', 'backlogs', 'internships', 'certifications', 'coding_skills', 'communication_skills', 'aptitude_score', 'projects', 'placed']


In [2]:
benchmark_features = [
    "cgpa",
    "coding_skills",
    "aptitude_score",
    "communication_skills",
    "internships",
    "certifications",
    "projects"
]

placed_students = df[df["placed"] == 1]

benchmarks = placed_students[benchmark_features].agg(["mean", "median"])

print("PLACED STUDENT BENCHMARKS")
print(benchmarks.round(2))

PLACED STUDENT BENCHMARKS
        cgpa  coding_skills  aptitude_score  communication_skills  \
mean    8.14           7.52           79.55                  5.48   
median  8.13           8.00           80.00                  6.00   

        internships  certifications  projects  
mean           1.53            2.53      2.45  
median         2.00            3.00      2.00  


In [5]:
reference_values = {
    "cgpa": benchmarks.loc["median", "cgpa"],
    "coding_skills": benchmarks.loc["median", "coding_skills"],
    "aptitude_score": benchmarks.loc["median", "aptitude_score"],
    "communication_skills": benchmarks.loc["median", "communication_skills"],
    "internships": benchmarks.loc["median", "internships"],
    "certifications": benchmarks.loc["median", "certifications"],
    "projects": benchmarks.loc["median", "projects"]
}

def calculate_skill_gaps(student):
    gaps = {}

    for feature, reference in reference_values.items():
        current = student[feature]
        gap = max(reference - current, 0)

        gaps[feature] = {
            "current": current,
            "reference": reference,
            "gap": gap
        }

    return gaps

In [6]:
student = df.iloc[0]

skill_gaps = calculate_skill_gaps(student)

for feature, values in skill_gaps.items():
    print(
        feature,
        " | Current:", values["current"],
        "| Reference:", values["reference"],
        "| Gap:", round(values["gap"], 2)
    )

cgpa  | Current: 8.4 | Reference: 8.13 | Gap: 0
coding_skills  | Current: 1 | Reference: 8.0 | Gap: 7.0
aptitude_score  | Current: 69 | Reference: 80.0 | Gap: 11.0
communication_skills  | Current: 3 | Reference: 6.0 | Gap: 3.0
internships  | Current: 2 | Reference: 2.0 | Gap: 0.0
certifications  | Current: 2 | Reference: 3.0 | Gap: 1.0
projects  | Current: 0 | Reference: 2.0 | Gap: 2.0


In [9]:
feature_ranges = {
    "cgpa": 10,
    "coding_skills": 10,
    "aptitude_score": 100,
    "communication_skills": 10,
    "internships": df["internships"].max(),
    "certifications": df["certifications"].max(),
    "projects": df["projects"].max()
}

for feature, values in skill_gaps.items():
    values["gap_percentage"] = round(
        (values["gap"] / feature_ranges[feature]) * 100, 2
    )

for feature, values in skill_gaps.items():
    print(
        feature,
        "| Gap:", round(values["gap"], 2),
        "| Gap %:", values["gap_percentage"], "%"
    )

cgpa | Gap: 0 | Gap %: 0.0 %
coding_skills | Gap: 7.0 | Gap %: 70.0 %
aptitude_score | Gap: 11.0 | Gap %: 11.0 %
communication_skills | Gap: 3.0 | Gap %: 30.0 %
internships | Gap: 0.0 | Gap %: 0.0 %
certifications | Gap: 1.0 | Gap %: 20.0 %
projects | Gap: 2.0 | Gap %: 40.0 %


In [10]:
importance_weights = {
    "coding_skills": 0.462618,
    "aptitude_score": 0.390068,
    "cgpa": 0.265157,
    "internships": 0.016561,
    "certifications": 0.006796,
    "communication_skills": 0.005526,
    "projects": 0.022528
}

for feature, values in skill_gaps.items():
    values["priority_score"] = round(
        values["gap_percentage"] * importance_weights[feature],
        2
    )

priority_table = pd.DataFrame([
    {
        "feature": feature,
        "current": values["current"],
        "reference": values["reference"],
        "gap": values["gap"],
        "gap_percentage": values["gap_percentage"],
        "priority_score": values["priority_score"]
    }
    for feature, values in skill_gaps.items()
])

priority_table = priority_table.sort_values(
    "priority_score",
    ascending=False
)

priority_table

,feature,current,reference,gap,gap_percentage,priority_score
1,coding_skills,1.0,8.00,7.0,70.0,32.38
2,aptitude_score,69.0,80.00,11.0,11.0,4.29
6,projects,0.0,2.00,2.0,40.0,0.90
3,communication_skills,3.0,6.00,3.0,30.0,0.17
5,certifications,2.0,3.00,1.0,20.0,0.14
0,cgpa,8.4,8.13,0.0,0.0,0.00
4,internships,2.0,2.00,0.0,0.0,0.00


In [11]:
recommendations = {
    "coding_skills": "Improve coding fundamentals, problem-solving, and regular programming practice.",
    "aptitude_score": "Practice quantitative aptitude, logical reasoning, and timed aptitude questions.",
    "cgpa": "Maintain your current academic performance and continue focusing on consistent semester results.",
    "communication_skills": "Improve communication through presentations, discussions, interviews, and speaking practice.",
    "internships": "Gain practical exposure through internships, industry projects, or relevant hands-on work.",
    "certifications": "Consider completing relevant certifications that strengthen your technical profile.",
    "projects": "Build practical projects that demonstrate your technical and problem-solving abilities."
}

priority_table["recommendation"] = priority_table["feature"].map(recommendations)

priority_table

,feature,current,reference,gap,gap_percentage,priority_score,recommendation
1,coding_skills,1.0,8.00,7.0,70.0,32.38,"Improve coding fundamentals, problem-solving, ..."
2,aptitude_score,69.0,80.00,11.0,11.0,4.29,"Practice quantitative aptitude, logical reason..."
6,projects,0.0,2.00,2.0,40.0,0.90,Build practical projects that demonstrate your...
3,communication_skills,3.0,6.00,3.0,30.0,0.17,"Improve communication through presentations, d..."
5,certifications,2.0,3.00,1.0,20.0,0.14,Consider completing relevant certifications th...
0,cgpa,8.4,8.13,0.0,0.0,0.00,Maintain your current academic performance and...
4,internships,2.0,2.00,0.0,0.0,0.00,"Gain practical exposure through internships, i..."


In [12]:
top_3 = priority_table[
    priority_table["priority_score"] > 0
].head(3)

print("TOP 3 IMPROVEMENT AREAS")
print()

for i, (_, row) in enumerate(top_3.iterrows(), start=1):
    print(f"{i}. {row['feature']}")
    print(f"   Current: {row['current']}")
    print(f"   Reference: {row['reference']}")
    print(f"   Gap: {row['gap_percentage']}%")
    print(f"   Recommendation: {row['recommendation']}")
    print()

TOP 3 IMPROVEMENT AREAS

1. coding_skills
   Current: 1.0
   Reference: 8.0
   Gap: 70.0%
   Recommendation: Improve coding fundamentals, problem-solving, and regular programming practice.

2. aptitude_score
   Current: 69.0
   Reference: 80.0
   Gap: 11.0%
   Recommendation: Practice quantitative aptitude, logical reasoning, and timed aptitude questions.

3. projects
   Current: 0.0
   Reference: 2.0
   Gap: 40.0%
   Recommendation: Build practical projects that demonstrate your technical and problem-solving abilities.



In [14]:
def analyze_skill_gap(student):
    results = []

    for feature, reference in reference_values.items():
        current = student[feature]
        gap = max(reference - current, 0)
        gap_percentage = (gap / feature_ranges[feature]) * 100
        priority_score = gap_percentage * importance_weights[feature]

        results.append({
            "feature": feature,
            "current": current,
            "reference": reference,
            "gap": round(gap, 2),
            "gap_percentage": round(gap_percentage, 2),
            "priority_score": round(priority_score, 2),
            "recommendation": recommendations[feature]
        })

    result = pd.DataFrame(results)
    result = result.sort_values("priority_score", ascending=False).reset_index(drop=True)

    return result

In [15]:
final_analysis = analyze_skill_gap(df.iloc[0])

final_analysis

,feature,current,reference,gap,gap_percentage,priority_score,recommendation
0,coding_skills,1.0,8.00,7.0,70.0,32.38,"Improve coding fundamentals, problem-solving, ..."
1,aptitude_score,69.0,80.00,11.0,11.0,4.29,"Practice quantitative aptitude, logical reason..."
2,projects,0.0,2.00,2.0,40.0,0.90,Build practical projects that demonstrate your...
3,communication_skills,3.0,6.00,3.0,30.0,0.17,"Improve communication through presentations, d..."
4,certifications,2.0,3.00,1.0,20.0,0.14,Consider completing relevant certifications th...
5,cgpa,8.4,8.13,0.0,0.0,0.00,Maintain your current academic performance and...
6,internships,2.0,2.00,0.0,0.0,0.00,"Gain practical exposure through internships, i..."
